# nanochat-ru SFT notebook (Kaggle T4 x2) -- phase 2 of training

Prerequisite: the pretraining run from `kaggle/kaggle_train.ipynb` must already be complete, with
`base_checkpoints/d4` and `tokenizer/` present on Google Drive (confirmed: 880/880 steps, min
validation bpb 1.099 -- see README.md).

This notebook fine-tunes that base (pure text completion) model into something that actually
responds to `<|user_start|>hi<|user_end|>` with a reply, instead of just continuing the pattern.

Upload this `.ipynb` directly via File -> Upload Notebook (same as the pretrain one) rather than
copying cells by hand. Requires the same 4 Kaggle Secrets as before (`GDRIVE_CLIENT_ID`,
`GDRIVE_CLIENT_SECRET`, `GDRIVE_OAUTH_TOKEN`, `GDRIVE_FOLDER_ID` -- see `docs/RCLONE_GDRIVE_SETUP.md`),
T4 x2 accelerator, and internet access.

## Cell 1: clone repo, install dependencies, rclone

In [6]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"
MODEL_TAG = "d4"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

# Install into the kernel's own Python (sys.executable), not an isolated .venv -- see
# kaggle_train.ipynb Cell 1 for why `uv sync` doesn't work for this notebook's execution model.
!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

Repo already present, pulling latest...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 6.01 KiB | 3.00 MiB/s, done.
From https://github.com/nadeko0/nanochat-ru
   af9a925..717bb4f  master     -> origin/master
Updating af9a925..717bb4f
Fast-forward
 .gitignore              |   4 +
 README.md               |  68 ++++++++++----
 kaggle/kaggle_sft.ipynb | 239 ++++++++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 294 insertions(+), 17 deletions(-)
 create mode 100644 kaggle/kaggle_sft.ipynb
Using Python 3.12.13 environment at: /usr
Resolved 84 packages in 264ms                                        
Checked 84 packages in 1ms
Cell 1 done.


## Cell 2: configure rclone, pull the pretrained base checkpoint + tokenizer from Drive

In [7]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

# Pull tokenizer + base checkpoint (required) and any existing SFT checkpoint (in case this
# notebook is being re-run and we want to know one already exists, e.g. to avoid overwriting
# without noticing).
for subdir in ["tokenizer", "base_checkpoints", "chatsft_checkpoints"]:
    remote_path = f"{DRIVE_REMOTE}{subdir}"
    local_path = os.path.join(NANOCHAT_BASE_DIR, subdir)
    listing = subprocess.run(["rclone", "lsf", remote_path], capture_output=True, text=True)
    if listing.returncode == 0 and listing.stdout.strip():
        print(f"Found {subdir} on Drive, downloading...")
        !rclone copy {remote_path} {local_path} --checksum -v
    else:
        print(f"No {subdir} on Drive yet.")

base_ckpt_dir = os.path.join(NANOCHAT_BASE_DIR, "base_checkpoints", MODEL_TAG)
assert os.path.isdir(base_ckpt_dir) and os.listdir(base_ckpt_dir), (
    f"No base checkpoint found at {base_ckpt_dir} -- run kaggle_train.ipynb (pretraining) first."
)
print(f"Base checkpoint ready at {base_ckpt_dir}")

existing_sft_dir = os.path.join(NANOCHAT_BASE_DIR, "chatsft_checkpoints", MODEL_TAG)
if os.path.isdir(existing_sft_dir) and os.listdir(existing_sft_dir):
    print(f"NOTE: an SFT checkpoint already exists at {existing_sft_dir} -- Cell 3 will overwrite it.")

           0 2026-08-10 16:16:45        -1 base_checkpoints
           0 2026-08-10 15:53:07        -1 base_data_climbmix
           0 2026-08-10 15:56:35        -1 tokenizer
Found tokenizer on Drive, downloading...
2026/08/10 17:42:35 INFO  : There was nothing to transfer
2026/08/10 17:42:35 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 2 / 2, 100%, Listed 4
Elapsed time:         0.1s

Found base_checkpoints on Drive, downloading...
2026/08/10 17:42:37 INFO  : There was nothing to transfer
2026/08/10 17:42:37 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                20 / 20, 100%, Listed 42
Elapsed time:         1.5s

No chatsft_checkpoints on Drive yet.
Base checkpoint ready at /kaggle/working/nanochat_cache/base_checkpoints/d4


## Cell 3: SFT (teaches the base model to behave like a chat assistant)

The base model only completes text -- it was never taught that
`<|user_start|>hi<|user_end|><|assistant_start|>` should be answered with something like "Hello!"
rather than just continuing the pattern. `scripts/chat_sft.py` fine-tunes it on conversation data
for that.

Skips MMLU/GSM8K in the training mixture (`--mmlu-epochs=0 --gsm8k-epochs=0`): at 36.7M params this
model has no realistic shot at multiple-choice reasoning or math, so including them would only burn
GPU time without helping the one thing we actually want -- coherent short replies.

Caps the run at `--num-iterations=500` for a different reason: unlike `base_train.py`,
`chat_sft.py` has **no `--save-every`** -- it only writes a checkpoint at the very end of the run.
A killed Kaggle session mid-SFT loses 100% of that session's progress, not just one save interval.
Bounding the run keeps a single session's risk small (~35-40 min at the pretraining run's ~59k
tok/sec) instead of gambling a full SmolTalk epoch (460K conversations) on the interruption
lottery. To do more SFT later, rerun this whole notebook with a higher `SFT_ITERATIONS` -- each run
starts fresh from the base checkpoint, not resumed from a previous SFT run.

In [8]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

SFT_ITERATIONS = 500

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}, log={SYNC_LOG}")

sft_cmd = (
    "torchrun --standalone --nproc_per_node=2 -m scripts.chat_sft -- "
    f"--model-tag={MODEL_TAG} --mmlu-epochs=0 --gsm8k-epochs=0 "
    f"--num-iterations={SFT_ITERATIONS} --chatcore-every=-1 --eval-every=100 --run=dummy"
)
print(f"Running: {sft_cmd}")
try:
    !{sft_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    print("SFT cell finished (or was interrupted), sync watcher stopped.")
    # chat_sft.py only saves at the very end -- if this was interrupted, there is no
    # partial checkpoint to sync, so a failed sync here just means "nothing new yet".
    !python kaggle/sync_checkpoints.py --remote gdrive: --once --log-file {SYNC_LOG}

Started background checkpoint sync watcher, pid=53776, log=/kaggle/working/sync_checkpoints.log
Running: torchrun --standalone --nproc_per_node=2 -m scripts.chat_sft -- --model-tag=d4 --mmlu-epochs=0 --gsm8k-epochs=0 --num-iterations=500 --chatcore-every=-1 --eval-every=100 --run=dummy
sync_checkpoints: base_dir=/kaggle/working/nanochat_cache remote=gdrive: interval=120s once=False
[2026-08-10 17:42:38] OK   tokenizer -> gdrive:/tokenizer
W0810 17:42:40.597000 53777 torch/distributed/run.py:803] 
W0810 17:42:40.597000 53777 torch/distributed/run.py:803] *****************************************
W0810 17:42:40.597000 53777 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0810 17:42:40.597000 53777 torch/distributed/run.py:803] *****************************************
[W810 17:42:40.427296739 socke

## Cell 4: quick chat test, right here in Kaggle

In [9]:
import os

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

!python -m scripts.chat_cli -i sft -g {MODEL_TAG} -p "hi"
!python -m scripts.chat_cli -i sft -g {MODEL_TAG} -p "What is your name?"

Autodetected device type: cuda
2026-08-10 17:58:54,661 - nanochat.common - INFO - Distributed world size: 1
2026-08-10 17:58:54,661 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d4 with step 125
2026-08-10 17:58:55,013 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 4, 'n_head': 2, 'n_kv_head': 2, 'n_embd': 256, 'window_pattern': 'L'}

NanoChat Interactive Mode
--------------------------------------------------
Type 'quit' or 'exit' to end the conversation
Type 'clear' to start a new conversation
--------------------------------------------------

Assistant: I'm glad you asked. As a new I'm excited to dive into the world of my favorite hobby. I love spending time with my loved ones, learning about the world, and finding ways to harness and communicate with others. I'll be happy to help you get started on this new hobby for you.<|assistant_end|>
Autode